# Nemotron-3-Nano-30B: SFT Training with LoRA (Rank 32)

Supervised fine-tuning on 8,569 chain-of-thought reasoning trajectories using LoRA rank 32 on the `in_proj`, `out_proj`, `up_proj`, and `down_proj` modules. Each training example follows a System/User/Assistant format where the assistant output includes a reasoning chain ending in `\boxed{answer}`.

**Kaggle inputs required:**
- Competition dataset: `nvidia-nemotron-3-reasoning-challenge`
- Model: `metric/nemotron-3-nano-30b-a3b-bf16`
- Private dataset: `nemotron-reasoning-data` (contains `train_reasoning_v5.jsonl`)

In [ ]:
# 1. Setup
import site
site.addsitedir(
    "/kaggle/usr/lib/notebooks/ryanholbrook/nvidia_utility_script/"
    "nvidia_cutlass_dsl/python_packages/"
)

import os, gc, json, warnings, logging
import torch
import mamba_ssm  # must precede transformers import

warnings.filterwarnings("ignore")
logging.getLogger("transformers").setLevel(logging.ERROR)
logging.getLogger("transformers_modules").setLevel(logging.ERROR)

print(f"PyTorch : {torch.__version__}")
print(f"CUDA    : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        gb = torch.cuda.get_device_properties(i).total_memory / 1e9
        print(f"  GPU {i}: {torch.cuda.get_device_name(i)} — {gb:.1f} GB")

In [ ]:
# 2. Configuration
LORA_RANK  = 32
LORA_ALPHA = 64
MAX_LENGTH = 1024
NUM_EPOCHS = 2
LR         = 2e-4
BATCH_SIZE = 1
GRAD_ACCUM = 8
SEED       = 42

OUTPUT_DIR      = "/kaggle/working"
MODEL_PATH      = "/kaggle/input/models/metric/nemotron-3-nano-30b-a3b-bf16/transformers/default/1"
REASONING_JSONL = "/kaggle/input/datasets/shamathmika/nemotron-reasoning-data/train_reasoning_v5.jsonl"
TRAIN_CSV       = "/kaggle/input/competitions/nvidia-nemotron-model-reasoning-challenge/train.csv"

print("Configuration ready.")

In [ ]:
# 3. Load data
import polars as pl
from collections import Counter

def load_examples():
    if os.path.exists(REASONING_JSONL):
        examples = []
        with open(REASONING_JSONL) as f:
            for line in f:
                line = line.strip()
                if line:
                    examples.append(json.loads(line))
        print(f"Loaded {len(examples)} examples from JSONL")
        return examples

    print("JSONL not found, falling back to train.csv")
    df = pl.read_csv(TRAIN_CSV)
    return [
        {"id": r["id"], "prompt": r["prompt"],
         "answer": str(r["answer"]), "reasoning": "", "task_type": ""}
        for r in df.to_dicts()
    ]

raw_examples = load_examples()
task_counts  = Counter(ex.get("task_type", "unknown") for ex in raw_examples)
print("\nTask distribution:")
for task, n in sorted(task_counts.items(), key=lambda x: -x[1]):
    print(f"  {task:<20} {n}")

In [ ]:
# 4. Format training examples
SYSTEM_PROMPTS = {
    "roman": (
        "You solve Roman numeral conversion puzzles. "
        "Study the examples, identify the mapping, and write your final answer in \\boxed{}."
    ),
    "unit_conversion": (
        "You solve unit conversion puzzles. "
        "Derive the conversion factor from the examples, apply it, "
        "and write your final numeric answer in \\boxed{}."
    ),
    "gravity": (
        "You solve physics puzzles with a hidden gravitational constant. "
        "Estimate g from d = 0.5·g·t² using the examples, compute the target, "
        "and write your final numeric answer in \\boxed{}."
    ),
    "cipher_text": (
        "You solve text cipher puzzles. "
        "Deduce the encryption rule from the examples, reverse it, "
        "and write the plaintext answer in \\boxed{}."
    ),
    "bit_manipulation": (
        "You solve 8-bit binary transformation puzzles. "
        "Identify the bitwise rule from the examples, apply it, "
        "and write your final 8-bit binary answer in \\boxed{}."
    ),
    "symbol_transform": (
        "You solve symbolic transformation puzzles. "
        "Identify the rule from the examples, apply it, "
        "and write your final answer in \\boxed{}."
    ),
}

GENERIC_SYSTEM = (
    "You are an expert reasoning assistant. "
    "Analyze the examples to identify the hidden pattern or rule, apply it to the query, "
    "and always place your final answer inside \\boxed{}."
)


def format_example(ex: dict) -> str:
    task_type = ex.get("task_type", "").strip().lower()
    system    = SYSTEM_PROMPTS.get(task_type, GENERIC_SYSTEM)
    prompt    = ex["prompt"].strip()
    answer    = str(ex["answer"]).strip()
    reasoning = ex.get("reasoning", "").strip()

    if reasoning:
        assistant = f"{reasoning}\n\nThe answer is \\boxed{{{answer}}}."
    else:
        assistant = f"The answer is \\boxed{{{answer}}}."

    return (
        f"System: {system}\n\n"
        f"User: {prompt}\n\n"
        f"Assistant: {assistant}"
    )


texts = [format_example(ex) for ex in raw_examples]
print(f"Formatted {len(texts)} training sequences.")
print("\nSample:")
print(texts[0][:600])

In [ ]:
# 5. Tokenizer
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

print(f"Vocab size : {tokenizer.vocab_size}")
print(f"EOS token  : {tokenizer.eos_token!r}  id={tokenizer.eos_token_id}")
print(f"Sample len : {len(tokenizer(texts[0])['input_ids'])} tokens")

In [ ]:
# 6. Dataset
from datasets import Dataset

def tokenize_batch(batch):
    enc = tokenizer(
        batch["text"],
        truncation=True,
        max_length=MAX_LENGTH,
        padding=False,
    )
    enc["labels"] = enc["input_ids"].copy()
    return enc

raw_ds       = Dataset.from_dict({"text": texts})
tokenized_ds = raw_ds.map(
    tokenize_batch,
    batched=True,
    batch_size=64,
    remove_columns=["text"],
    desc="Tokenising",
)

lengths   = sorted(len(x) for x in tokenized_ds["input_ids"])
pct_trunc = sum(1 for x in lengths if x >= MAX_LENGTH) / len(lengths) * 100
print(f"Examples  : {len(tokenized_ds)}")
print(f"Lengths   : min={lengths[0]}  median={lengths[len(lengths)//2]}  max={lengths[-1]}")
print(f"Truncated : {pct_trunc:.1f}%")

In [ ]:
# 7. Load model
import gc
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
from transformers import AutoModelForCausalLM

os.makedirs("/tmp/offload", exist_ok=True)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH,
    device_map="auto",
    dtype=torch.bfloat16,
    offload_folder="/tmp/offload",
    trust_remote_code=True,
)

gc.collect()
torch.cuda.empty_cache()
for i in range(torch.cuda.device_count()):
    alloc = torch.cuda.memory_allocated(i) / 1e9
    total = torch.cuda.get_device_properties(i).total_memory / 1e9
    print(f"GPU {i}: {alloc:.1f} GB / {total:.1f} GB")
print("Model loaded.")

In [ ]:
# 8. LoRA adapter
from peft import LoraConfig, get_peft_model, TaskType

lora_config = LoraConfig(
    r=LORA_RANK,
    lora_alpha=LORA_ALPHA,
    target_modules=r".*\.(in_proj|out_proj|up_proj|down_proj)$",
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

In [ ]:
# 9. Triton ptxas fix (Blackwell GPU requires ptxas-blackwell)
import shutil, stat

bin_dir = "/kaggle/usr/lib/notebooks/ryanholbrook/nvidia_utility_script/triton/backends/nvidia/bin"

ptxas_dst = "/tmp/ptxas_exec"
shutil.copy2(f"{bin_dir}/ptxas", ptxas_dst)
os.chmod(ptxas_dst, os.stat(ptxas_dst).st_mode | stat.S_IEXEC | stat.S_IXGRP | stat.S_IXOTH)
os.environ["TRITON_PTXAS_PATH"] = ptxas_dst

ptxas_bw_dst = "/tmp/ptxas_blackwell_exec"
shutil.copy2(f"{bin_dir}/ptxas-blackwell", ptxas_bw_dst)
os.chmod(ptxas_bw_dst, os.stat(ptxas_bw_dst).st_mode | stat.S_IEXEC | stat.S_IXGRP | stat.S_IXOTH)
os.environ["TRITON_PTXAS_BLACKWELL_PATH"] = ptxas_bw_dst

print("ptxas:", ptxas_dst)
print("ptxas-blackwell:", ptxas_bw_dst)

In [ ]:
# 10. Train
from transformers import TrainingArguments, Trainer, DataCollatorForLanguageModeling

data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False,
    pad_to_multiple_of=8,
)

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LR,
    lr_scheduler_type="cosine",
    warmup_steps=50,
    bf16=True,
    fp16=False,
    logging_steps=20,
    save_strategy="no",
    dataloader_num_workers=2,
    remove_unused_columns=False,
    seed=SEED,
    report_to="none",
    optim="adamw_torch_fused",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_ds,
    data_collator=data_collator,
)

print("Starting training...")
train_result = trainer.train()
print(f"Final loss: {train_result.training_loss:.4f}")

In [ ]:
# 11. Save adapter
model.save_pretrained(OUTPUT_DIR)

for fname in ["adapter_config.json", "adapter_model.safetensors"]:
    path   = os.path.join(OUTPUT_DIR, fname)
    exists = os.path.exists(path)
    size   = os.path.getsize(path) / 1e6 if exists else 0
    print(f"  {'ok' if exists else 'MISSING'}  {fname}  ({size:.1f} MB)")

In [ ]:
# 12. Sanity check
model.eval()

TEST_PROMPT = (
    "System: You solve Roman numeral conversion puzzles. "
    "Study the examples, identify the mapping, and write your final answer in \\boxed{}.\n\n"
    "User: In Alice's Wonderland, numbers are secretly converted into a different numeral system.\n"
    "11 -> XI\n15 -> XV\n94 -> XCIV\n19 -> XIX\n"
    "Now, write the number 38 in the Wonderland numeral system.\n\n"
    "Assistant:"
)

inputs = tokenizer(TEST_PROMPT, return_tensors="pt").to(next(model.parameters()).device)
with torch.no_grad():
    out = model.generate(
        **inputs,
        max_new_tokens=80,
        do_sample=False,
        eos_token_id=tokenizer.eos_token_id,
    )
response = tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
print("Output  :", response)
print("Expected: ... \\boxed{XXXVIII}")

In [ ]:
# 13. Package submission
import subprocess

result = subprocess.run(
    "zip -m submission.zip adapter_config.json adapter_model.safetensors",
    shell=True, check=True, capture_output=True, text=True, cwd=OUTPUT_DIR,
)
print(result.stdout or result.stderr)
print(f"submission.zip: {os.path.getsize(os.path.join(OUTPUT_DIR, 'submission.zip'))/1e6:.1f} MB")